# 类 BiLSTM-CRF 实现序列标注

2019-01-15
`NLP`, `TF`, `NER`

本文将基于几篇近年来 BiLSTM 与 CRF 做 NER 的论文，结合具体的 `Tensorflow` 代码，理解常见的深度学习序列标注方法。

## 序列标注


序列标注问题是自然语言处理中的基本问题之一，在深度学习火起来之前，常见的序列标注问题的解决方案都是借助于HMM模型，最大熵模型，CRF模型。尤其是CRF，是解决序列标注问题的主流方法。

随着深度学习的发展，RNN在序列标注问题中取得了巨大的成果。而且深度学习中的 end-to-end，也让序列标注问题变得更简单了。

序列标注问题包括自然语言处理中的分词，词性标注，命名实体识别，关键词抽取，词义角色标注等等。我们只要在做序列标注时给定特定的标签集合，就可以进行序列标注。

## BiLSTM-CRF

论文链接：_[Bidirectional LSTM-CRF Models for Sequence Tagging](https://arxiv.org/abs/1508.01991)_

经典的 BiLSTM-CRF 模型结构不复杂，双向的 LSTM 可以更好地刻画同一时刻上下文（前文与后文）对当前状态的影响，而 CRF 则在句子级别对 tag 序列进行约束。

![BiLSTM-CRF 模型](assets/bilstm_crf.jpeg)

值得注意的是模块的输入可以是 token 的 one-hot 编码或 embedding 或对应的稀疏特征。

最终，在参数 

$$\tilde{\theta}=\theta \cup \{[A]_{i,j} \forall i,j\}$$ 

（其中$\theta$ 表示 LSTM 模块的网络参数，$[A]_{i,j}$ 表示Tag从第i个值到第j个值的转移值，转移矩阵由 CRF 算得）下，句子 $[X]_1^T$ 对应标签序列 $[i]_1^T$ 的得分为：

$$s([x]_1^T, [i]_1^T, \tilde{\theta})=\sum _{t=1}^T ([A]_{[i]_{t-1},[i]_t}+[f_{\theta}]_{[i]_t,t})$$

模型训练步骤如下：

![BiLSTM-CRF 模型训练步骤](assets/bilstm_crf_steps.jpeg)

下面以 token 的 Embedding 作为输入的训练过程代码片断：

```python

# Word Embeddings
word_ids = vocab_words.lookup(words)
glove = np.load(params['glove'])['embeddings']  # np.array
variable = np.vstack([glove, [[0.]*params['dim']]])  # 注意这里的最后一维是零向量
variable = tf.Variable(variable, dtype=tf.float32, trainable=False)
embeddings = tf.nn.embedding_lookup(variable, word_ids)
embeddings = tf.layers.dropout(embeddings, rate=dropout, training=training)  # Embedding 作 dropout

```


```python
# LSTM
t = tf.transpose(embeddings, perm=[1, 0, 2])
lstm_cell_fw = tf.contrib.rnn.LSTMBlockFusedCell(params['lstm_size'])
lstm_cell_bw = tf.contrib.rnn.LSTMBlockFusedCell(params['lstm_size'])
lstm_cell_bw = tf.contrib.rnn.TimeReversedFusedRNN(lstm_cell_bw)
output_fw, _ = lstm_cell_fw(t, dtype=tf.float32, sequence_length=nwords)
output_bw, _ = lstm_cell_bw(t, dtype=tf.float32, sequence_length=nwords)
output = tf.concat([output_fw, output_bw], axis=-1)
output = tf.transpose(output, perm=[1, 0, 2])
output = tf.layers.dropout(output, rate=dropout, training=training)
```

参考文献：

0. [Bidirectional LSTM-CRF Models for Sequence Tagging](https://arxiv.org/abs/1508.01991) by Huang, Xu and Yu
1. [Neural Architectures for Named Entity Recognition](https://arxiv.org/abs/1603.01360) by Lample et al.
2. [End-to-end Sequence Labeling via Bi-directional LSTM-CNNs-CRF](https://arxiv.org/abs/1603.01354) by Ma et Hovy
3. [guillaumegenthial/tf_ner](https://github.com/guillaumegenthial/tf_ner) by Guillaume Genthial